In [1]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=./infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

foundry_endpoint = ! terraform -chdir=./infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=./infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=./infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

sessionpool_management_endpoint_python = ! terraform -chdir=./infra output -raw sessionpool_management_endpoint_python
sessionpool_management_endpoint_python = sessionpool_management_endpoint_python.n
print("Session Pool Management Endpoint:", sessionpool_management_endpoint_python)

sessionpool_mcp_endpoint_python = ! terraform -chdir=./infra output -raw sessionpool_mcp_endpoint_python
sessionpool_mcp_endpoint_python = sessionpool_mcp_endpoint_python.n
print("MCP Session Pool Endpoint:", sessionpool_mcp_endpoint_python)

sessionpool_management_endpoint_shell = ! terraform -chdir=./infra output -raw sessionpool_management_endpoint_shell
sessionpool_management_endpoint_shell = sessionpool_management_endpoint_shell.n
print("Session Pool Management Endpoint (Shell):", sessionpool_management_endpoint_shell)

sessionpool_mcp_endpoint_shell = ! terraform -chdir=./infra output -raw sessionpool_mcp_endpoint_shell
sessionpool_mcp_endpoint_shell = sessionpool_mcp_endpoint_shell.n
print("MCP Session Pool Endpoint (Shell):", sessionpool_mcp_endpoint_shell)

LLM Endpoint: ╷
│ Error: Output "aca_gemma4_31b_it_a100_fqdn" not found
│ 
│ The output variable requested could not be found in the state file. If you
│ recently added this to your configuration, be sure to run `terraform
│ apply`, since the state won't be updated with new output variables until
│ that command is run.
╵
Foundry Endpoint: https://foundry-400.cognitiveservices.azure.com/
Foundry API Key: AAACOGYQ3t...
LLM Model Deployment Name (ChatGPT): gpt-5.4
Session Pool Management Endpoint: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-agents-langchain-400/sessionPools/acasessionpool-python
MCP Session Pool Endpoint: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-agents-langchain-400/sessionPools/acasessionpool-python/mcp
Session Pool Management Endpoint (Shell): https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d70

In [2]:
%pip install langchain langchain-openai langchain-mcp-adapters langchain-azure-dynamic-sessions

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage

# model = ChatOpenAI(
#     base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
#     api_key="EMPTY",
#     model="google/gemma-4-31B-it",
#     streaming=True,
#     max_completion_tokens= 512
# )

model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name_chatgpt,
    streaming=True,
    max_completion_tokens=512
)

response = model.stream([HumanMessage(content="Tell me about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI.

I can help with things like:
- answering questions
- explaining concepts
- writing and editing
- brainstorming ideas
- summarizing information
- coding help
- math and logic
- planning, research, and organization

A few useful things to know about me:
- I generate responses based on patterns in data I was trained on.
- I don’t have feelings, personal experiences, or consciousness.
- I can sound confident even when I’m wrong, so it’s good to verify important information.
- My knowledge may be incomplete or outdated in some areas unless I’m given current information.
- I adapt to your style: you can ask for short, detailed, formal, casual, technical, or beginner-friendly answers.

If you want, I can also tell you:
- what I’m good at
- my limitations
- how to get better results from me
- a more personal/fun version of this intro

In [3]:
from langchain.agents import create_agent
from langchain_azure_dynamic_sessions.tools import SessionsPythonREPLTool
from azure.identity import AzureCliCredential

credential = AzureCliCredential()

def access_token_provider():
    token = credential.get_token("https://dynamicsessions.io/.default")
    return token.token

# get the management endpoint from the session pool in the Azure portal
toolPythonSession = SessionsPythonREPLTool(
    pool_management_endpoint=sessionpool_management_endpoint_python,
    access_token_provider=access_token_provider,
)

agent = create_agent(model=model, tools=[toolPythonSession])

async for step in agent.astream(
    {"messages": [{"role": "user", "content": "What is the current time in Tunisia and France ? Use the python REPL tool to get the answer."}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

What is the current time in Tunisia and France ? Use the python REPL tool to get the answer.
================================== Ai Message ==================================
Tool Calls:
  Python_REPL (call_gcyTp92uYaryO7HVqc9zokx1)
 Call ID: call_gcyTp92uYaryO7HVqc9zokx1
  Args:
    python_code: from datetime import datetime
from zoneinfo import ZoneInfo
now_tunisia = datetime.now(ZoneInfo('Africa/Tunis'))
now_france = datetime.now(ZoneInfo('Europe/Paris'))
print('Tunisia:', now_tunisia.strftime('%Y-%m-%d %H:%M:%S %Z%z'))
print('France:', now_france.strftime('%Y-%m-%d %H:%M:%S %Z%z'))
================================= Tool Message =================================
Name: Python_REPL

{
  "result": "",
  "stdout": "Tunisia: 2026-06-06 22:30:47 CET+0100\nFrance: 2026-06-06 23:30:47 CEST+0200\n",
  "stderr": ""
}
================================== Ai Message ==================================

Current local t

In [4]:
import io
import json

data = {"important_data": [1, 10, -1541]}
binary_io = io.BytesIO(json.dumps(data).encode("ascii"))

upload_metadata = toolPythonSession.upload_file(
    data=binary_io, remote_file_path="important_data.json"
)

code = f"""
import json

with open("{upload_metadata.full_path}") as f:
    data = json.load(f)

sum(data['important_data'])
"""

toolPythonSession.execute(code)

{'$id': '2',
 'status': 'Success',
 'stdout': '',
 'stderr': '',
 'result': -1530,
 'executionTimeInMilliseconds': 6}

In [5]:
from langchain_azure_dynamic_sessions.tools import SessionsBashTool

# get the management endpoint from the session pool in the Azure portal
toolBashSession = SessionsBashTool(
    pool_management_endpoint=sessionpool_management_endpoint_shell,
    access_token_provider=access_token_provider,
)

response = toolBashSession.execute("echo Hello from the session!")

print(json.dumps(response, indent=2))

{
  "identifier": "d1e6faa3-8eea-4ed5-9cc2-5f8484bc9d41",
  "status": "0",
  "result": {
    "stdout": "Hello from the session!\n",
    "stderr": "",
    "executionTimeInMilliseconds": 1
  }
}


In [6]:
response = toolBashSession.execute("printenv")

print(json.dumps(response, indent=2))

{
  "identifier": "d1e6faa3-8eea-4ed5-9cc2-5f8484bc9d41",
  "status": "0",
  "result": {
    "stdout": "PWD=/mnt/data\nENABLE_EGRESS=false\n_=/usr/bin/printenv\nHOME=/root\nCODE_INTERPRETER_CONFIG_FILE_PATH=/app/codeInterpreterConf.yaml\nSHLVL=1\nPATH=/usr/local/sbin:/usr/local/bin:/usr/sbin:/usr/bin:/sbin:/bin\n",
    "stderr": "",
    "executionTimeInMilliseconds": 3
  }
}


In [7]:
response = toolBashSession.execute("cat /app/codeInterpreterConf.yaml")

print(json.dumps(response, indent=2))

{
  "identifier": "d1e6faa3-8eea-4ed5-9cc2-5f8484bc9d41",
  "status": "0",
  "result": {
    "stdout": "kind: CodeInterpreterConfig\napiVersion: v1\nmetadata:\n  name: code-interpreter\nspec:\n  logging:\n    logLevel: 1 # 0: debug, 1: info, 2: warn, 3: error\n    useISO8601DateFormat: false\n  kernel:\n    enabled: false\n    isFirstParty: true\n    port: 6003\n\n  healthProbe: # It will respond back /health endpoint with 200 status code if the container is healthy\n    initialDelaySeconds: 0\n\n  fileOperation:\n    maxUploadFileSizeLimitInMBs: 250\n    enableFullPathAccess: true\n",
    "stderr": "",
    "executionTimeInMilliseconds": 2
  }
}


In [ ]:
response = toolBashSession.execute("curl ifconfig.me/ip")

print(json.dumps(response, indent=2))

{
  "identifier": "d1e6faa3-8eea-4ed5-9cc2-5f8484bc9d41",
  "status": "0",
  "result": {
    "stdout": "74.241.176.35",
    "stderr": "  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current\n                                 Dload  Upload   Total   Spent    Left  Speed\n\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0\r100    13  100    13    0     0     60      0 --:--:-- --:--:-- --:--:--    60\n",
    "executionTimeInMilliseconds": 280
  }
}


In [11]:
agent = create_agent(model=model, tools=[toolBashSession])

async for step in agent.astream(
    {"messages": [{"role": "user", "content": "Create a hello world flask app in a new remote environment. Then send a request to the app to show that it works."}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================

Create a hello world flask app in a new remote environment. Then send a request to the app to show that it works.
================================== Ai Message ==================================

I’ll set up a minimal Flask app in this environment, install Flask if needed, run the server in the background, and then curl it to verify it works.
Tool Calls:
  Bash (call_1kHnRusuqCiaNtDtF9v5EVJq)
 Call ID: call_1kHnRusuqCiaNtDtF9v5EVJq
  Args:
    bash_command: set -e
mkdir -p /tmp/flask_hello_app
cd /tmp/flask_hello_app
cat > app.py <<'PY'
from flask import Flask
app = Flask(__name__)

@app.route('/')
def hello():
    return 'Hello, World!'

if __name__ == '__main__':
    app.run(host='127.0.0.1', port=5000)
PY
python3 -m venv .venv
. .venv/bin/activate
python -m pip install --upgrade pip >/dev/null
pip install flask >/tmp/flask_install.log 2>&1
nohup python app.py >/tmp/flask_app.log 2>&1 &
APP_PID=$!
echo 

In [9]:
agent = create_agent(model=model, tools=[toolBashSession])

async for step in agent.astream(
    {"messages": [{"role": "user", 
                   "content": """
                        Get the history of the Microsoft stock price for the last 5 years and plot it. 
                        Install the required packages in the session if needed.
                        save the chart as `msft_5y.png`.
                   """}]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================


                        Get the history of the Microsoft stock price for the last 5 years and plot it. 
                        Install the required packages in the session if needed.
                        save the chart as `msft_5y.png`.
                   
================================== Ai Message ==================================
Tool Calls:
  Bash (call_pbTr767pSVIXrN2m6f2TrGg6)
 Call ID: call_pbTr767pSVIXrN2m6f2TrGg6
  Args:
    bash_command: python - <<'PY'
import importlib.util, subprocess, sys, os
mods=['yfinance','pandas','matplotlib']
missing=[m for m in mods if importlib.util.find_spec(m) is None]
if missing:
    subprocess.check_call([sys.executable,'-m','pip','install',*missing])
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime, timedelta

end = datetime.today()
start = end - timedelta(days=365*5+10)

df = yf.download('MSFT', star

In [10]:
%pip install Pillow

# upload a file to the session
# toolBashSession.upload_file(local_file_path="./rg.tf", remote_file_path="/mnt/user/rg.tf")

# list files in the session
files = toolBashSession.list_files()

# print files in the session
print(files)

# download a file from the session
toolBashSession.download_file(remote_file_path="msft_5y.png", local_file_path="./msft_5y.png")

# view the image in the notebook
from PIL import Image
image = Image.open("./msft_5y.png")
image.show()


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.
[]


HTTPError: 404 Client Error: Not Found for url: https://swedencentral.dynamicsessions.io/subscriptions/dcef7009-6b94-4382-afdc-17eb160d709a/resourceGroups/rg-ai-agents-langchain-400/sessionPools/acasessionpool-shell/files/msft_5y.png/content?identifier=d1e6faa3-8eea-4ed5-9cc2-5f8484bc9d41&api-version=2025-02-02-preview